<a href="https://colab.research.google.com/github/gunnsmart/upscaly-colab-version-/blob/arena%2F01a0fb33-upscaly-colab-version/upscayl_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Upscayl Studio · Google Colab

นี่คือ Web UI จริงสำหรับ Upscayl — Notebook จะเปิดหน้า **Gradio Studio** ให้ใช้งานใน browser มีแผงเลือกรูป/โมเดล และพื้นที่แสดงผลก่อน–หลัง ไม่ใช่แค่ code cell หรือ widget ใน Notebook

1. เลือก **Runtime → Change runtime type → GPU**
2. ถ้าต้องการเก็บไฟล์บน Drive ให้ตั้ง `USE_GOOGLE_DRIVE = True` ในเซลล์ workspace
3. เลือก **Runtime → Run all** แล้วรอ backend กับลิงก์ UI
4. เปิดลิงก์ Gradio ที่ได้; ใช้ username/password ซึ่งแสดงใน output ของเซลล์ UI
5. อัปโหลดภาพ เลือกค่าต่าง ๆ แล้วกด **เริ่ม Upscale**

ลิงก์ UI ชั่วคราวและมี password; อย่าแชร์ข้อมูลล็อกอินหรือใช้กับภาพลับ และหยุด Colab runtime เมื่อเลิกใช้งาน


In [ ]:
from pathlib import Path

# ตั้งเป็น True เพื่อเก็บ input/output ไว้ใน Google Drive
USE_GOOGLE_DRIVE = False
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    WORKSPACE = Path("/content/drive/MyDrive/Upscayl-Colab")
else:
    WORKSPACE = Path("/content/upscayl-colab")

INPUT_DIR = WORKSPACE / "input"
OUTPUT_DIR = WORKSPACE / "output"
RUNTIME_DIR = Path("/content/upscayl-colab-runtime")
BIN_DIR = RUNTIME_DIR / "bin"
BIN_PATH = BIN_DIR / "upscayl-bin"
MODEL_DIR = RUNTIME_DIR / "models"  # Upscayl backend ต้องการโฟลเดอร์ชื่อ models
for folder in (INPUT_DIR, OUTPUT_DIR, BIN_DIR, MODEL_DIR):
    folder.mkdir(parents=True, exist_ok=True)

print(f"Workspace: {WORKSPACE}")
print(f"Output จะถูกเก็บที่: {OUTPUT_DIR}")


## เตรียม Upscayl backend

เซลล์นี้ติดตั้ง dependency, ตรวจว่า Vulkan เห็น NVIDIA GPU และดาวน์โหลด CLI ทางการพร้อมตรวจ SHA-256 หาก runtime ใช้งาน Vulkan ไม่ได้ Notebook จะเก็บ error ไว้ แล้วเปิด UI เพื่อแจ้งสาเหตุแทน


In [ ]:
BACKEND_READY = False
BACKEND_ERROR = None
try:
    # เตรียม NVIDIA Vulkan และดาวน์โหลด CLI ทางการของ Upscayl
    import glob
    import hashlib
    import json
    import os
    import re
    import shutil
    import stat
    import subprocess
    import urllib.request
    import zipfile
    from pathlib import Path


    def run_command(args, *, check=True, env=None):
        result = subprocess.run(args, text=True, capture_output=True, env=env)
        if check and result.returncode != 0:
            detail = (result.stdout or "") + (result.stderr or "")
            raise RuntimeError(
                f"คำสั่งล้มเหลว: {' '.join(map(str, args))}\n{detail[-5000:]}"
            )
        return result


    # อ่านเวอร์ชันไดรเวอร์ เพื่อติดตั้ง NVIDIA Vulkan ICD รุ่นเดียวกับ driver branch
    smi = run_command(
        ["nvidia-smi", "--query-gpu=driver_version,name", "--format=csv,noheader"],
        check=False,
    )
    if smi.returncode != 0 or not smi.stdout.strip():
        raise RuntimeError(
            "ไม่พบ NVIDIA GPU ใน runtime นี้ กรุณาเลือก Runtime → Change runtime type → GPU "
            "แล้วรันเซลล์ใหม่"
        )

    gpu_line = smi.stdout.strip().splitlines()[0]
    parts = [part.strip() for part in gpu_line.split(",", 1)]
    driver_version = parts[0]
    gpu_name = parts[1] if len(parts) > 1 else "NVIDIA GPU"
    major_match = re.match(r"(\d+)", driver_version)
    if not major_match:
        raise RuntimeError(f"อ่าน NVIDIA driver version ไม่ได้: {driver_version}")
    driver_major = major_match.group(1)
    print(f"NVIDIA GPU: {gpu_name} | driver {driver_version}")

    apt_update = run_command(["apt-get", "update", "-qq"], check=False)
    if apt_update.returncode != 0:
        raise RuntimeError(f"apt-get update ล้มเหลว:\n{apt_update.stderr[-3000:]}")


    def apt_package_available(package):
        policy = run_command(["apt-cache", "policy", package], check=False).stdout
        match = re.search(r"Candidate:\s*(\S+)", policy)
        return bool(match and match.group(1) != "(none)")


    packages = ["libvulkan1", "vulkan-tools"]
    matching_driver_package = f"libnvidia-gl-{driver_major}"
    if apt_package_available(matching_driver_package):
        packages.append(matching_driver_package)
    else:
        print(f"ไม่พบแพ็กเกจ {matching_driver_package} ใน apt; จะลองใช้ Vulkan ICD ที่มีอยู่ใน runtime")
    if apt_package_available("libomp5"):
        packages.append("libomp5")

    apt_install = run_command(["apt-get", "install", "-y", "-qq", *packages], check=False)
    if apt_install.returncode != 0:
        raise RuntimeError(
            f"ติดตั้ง Vulkan dependencies ไม่สำเร็จ ({', '.join(packages)}):\n"
            f"{apt_install.stdout[-2000:]}\n{apt_install.stderr[-3000:]}"
        )

    # บังคับใช้ NVIDIA ICD หากแพ็กเกจติดตั้งไฟล์ JSON ไว้แล้ว
    icd_candidates = []
    for icd_root in ("/usr/share/vulkan/icd.d", "/etc/vulkan/icd.d"):
        icd_candidates.extend(glob.glob(f"{icd_root}/*nvidia*.json"))
    NVIDIA_ICD = None
    for candidate in sorted(icd_candidates):
        try:
            with open(candidate, "r", encoding="utf-8") as handle:
                if "ICD" in json.load(handle):
                    NVIDIA_ICD = candidate
                    break
        except (OSError, json.JSONDecodeError):
            continue
    if NVIDIA_ICD:
        os.environ["VK_ICD_FILENAMES"] = NVIDIA_ICD
        os.environ["VK_DRIVER_FILES"] = NVIDIA_ICD
        print(f"NVIDIA Vulkan ICD: {NVIDIA_ICD}")
    else:
        print("ไม่พบไฟล์ NVIDIA ICD JSON; จะตรวจสอบ Vulkan device ที่ runtime เปิดให้ใช้")

    vulkan = run_command(["vulkaninfo", "--summary"], check=False, env=os.environ.copy())
    vulkan_report = (vulkan.stdout or "") + (vulkan.stderr or "")
    if vulkan.returncode != 0:
        raise RuntimeError(
            "Vulkan เริ่มทำงานไม่ได้ใน Colab runtime นี้\n"
            f"{vulkan_report[-5000:]}\n"
            "ตรวจว่าเลือก GPU runtime แล้ว restart runtime จากนั้นลองใหม่ "
            "(Colab บาง runtime อาจไม่มี Vulkan support)"
        )

    has_nvidia_device = bool(re.search(r"vendorID\s*=\s*0x10de\b", vulkan_report, re.IGNORECASE))
    if not has_nvidia_device:
        raise RuntimeError(
            "vulkaninfo ไม่พบ NVIDIA device (vendor ID 0x10de) จึงยังใช้ Upscayl NCNN/Vulkan ไม่ได้\n"
            f"{vulkan_report[:3000]}\n"
            "ลองเลือก GPU runtime ใหม่และ restart; การมี CUDA/nvidia-smi อย่างเดียวไม่ยืนยันว่า Vulkan ใช้งานได้"
        )
    print("✅ Vulkan ตรวจพบ NVIDIA GPU")


    def download_file(url, destination, expected_sha256=None):
        destination = Path(destination)
        destination.parent.mkdir(parents=True, exist_ok=True)
        if destination.is_file() and destination.stat().st_size > 0:
            if expected_sha256 is None:
                return destination
            current_hash = hashlib.sha256(destination.read_bytes()).hexdigest()
            if current_hash == expected_sha256:
                return destination
        temporary = destination.with_name(destination.name + ".part")
        request = urllib.request.Request(
            url, headers={"User-Agent": "upscayl-colab-notebook/1.0"}
        )
        try:
            with urllib.request.urlopen(request, timeout=180) as response, temporary.open("wb") as output:
                shutil.copyfileobj(response, output)
            if expected_sha256:
                actual_hash = hashlib.sha256(temporary.read_bytes()).hexdigest()
                if actual_hash.lower() != expected_sha256.lower():
                    raise RuntimeError(
                        f"SHA-256 ไม่ตรงสำหรับ {destination.name}: {actual_hash}"
                    )
            temporary.replace(destination)
        finally:
            if temporary.exists():
                temporary.unlink()
        return destination


    # Pin release เพื่อให้ได้ไบนารีที่ตรวจสอบ SHA-256 ได้
    UPSCAYL_CLI_TAG = "20251207-174704"
    CLI_URL = (
        "https://github.com/upscayl/upscayl-ncnn/releases/download/"
        f"{UPSCAYL_CLI_TAG}/upscayl-bin-{UPSCAYL_CLI_TAG}-linux.zip"
    )
    CLI_SHA256 = "a9fab3c770b62f2b7a35d8d6d61eb4e8b3aef79128b665c919d080b85a2292f2"
    CLI_ARCHIVE = RUNTIME_DIR / f"upscayl-bin-{UPSCAYL_CLI_TAG}-linux.zip"
    download_file(CLI_URL, CLI_ARCHIVE, CLI_SHA256)

    extract_dir = RUNTIME_DIR / "upscayl-bin-extracted"
    extract_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(CLI_ARCHIVE) as archive:
        root = extract_dir.resolve()
        for member in archive.infolist():
            target = (extract_dir / member.filename).resolve()
            if not target.is_relative_to(root):
                raise RuntimeError(f"พบ path ที่ไม่ปลอดภัยใน ZIP: {member.filename}")
        archive.extractall(extract_dir)

    executables = [path for path in extract_dir.rglob("upscayl-bin") if path.is_file()]
    if not executables:
        raise RuntimeError("ไม่พบไฟล์ upscayl-bin ใน official Linux release archive")
    shutil.copy2(executables[0], BIN_PATH)
    BIN_PATH.chmod(BIN_PATH.stat().st_mode | stat.S_IXUSR | stat.S_IXGRP | stat.S_IXOTH)

    link_check = run_command(["ldd", str(BIN_PATH)], check=False)
    missing_libraries = [line.strip() for line in link_check.stdout.splitlines() if "not found" in line]
    if missing_libraries:
        raise RuntimeError("ไลบรารีที่ CLI ต้องใช้ยังไม่ครบ:\n" + "\n".join(missing_libraries))

    help_check = run_command([str(BIN_PATH), "-h"], check=False, env=os.environ.copy())
    if "Usage:" not in ((help_check.stdout or "") + (help_check.stderr or "")):
        print("⚠️ ดาวน์โหลด CLI แล้ว แต่ตรวจข้อความ help ไม่ได้; จะลองทำงานต่อ")

    print(f"✅ Upscayl NCNN CLI พร้อมใช้งาน: {BIN_PATH}")
    BACKEND_READY = True
    if callable(globals().get("set_status")):
        set_status(f"Upscayl พร้อมใช้งานบน {gpu_name}", "success")
    if globals().get("header") is not None:
        import html as _html
        header.value = header.value.replace("ตรวจสอบสถานะ backend ด้านล่าง", _html.escape(str(gpu_name)))
except Exception as setup_error:
    BACKEND_ERROR = f"{type(setup_error).__name__}: {setup_error}"
    print(f"⚠️ Upscayl backend ยังไม่พร้อม: {BACKEND_ERROR}")
    print("หน้า UI จะยังแสดง แต่ต้องแก้การตั้งค่า GPU/Vulkan ก่อนเริ่มประมวลผล")
    if callable(globals().get("set_status")):
        set_status(f"UI พร้อมแล้ว แต่ backend ยังไม่พร้อม: {BACKEND_ERROR}", "error")


## เปิด Web UI

เมื่อรันเซลล์นี้ Colab จะเปิด Gradio Web UI แบบชั่วคราวและป้องกันด้วยรหัสผ่านที่สร้างใหม่ในแต่ละ runtime มีตัวเลือกโมเดล/scale/format/tile, TTA, อัปโหลดหลายภาพ, Drive input, progress, ภาพเปรียบเทียบ และ ZIP download


In [ ]:
# Launch a real browser-based Upscayl Studio web UI from Colab.
import datetime
import os
import secrets
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

try:
    import gradio as gr
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "gradio"])
    import gradio as gr

MODEL_LABELS = {
    "upscayl-standard-4x": "Standard — ภาพทั่วไป (แนะนำ)",
    "upscayl-lite-4x": "Lite — โมเดลเล็ก/เร็ว",
    "high-fidelity-4x": "High Fidelity — เน้นรายละเอียด",
    "remacri-4x": "Remacri — รายละเอียดและ texture",
    "ultramix-balanced-4x": "Ultramix Balanced — สมดุล",
    "ultrasharp-4x": "Ultrasharp — เน้นความคม",
    "digital-art-4x": "Digital Art — ภาพวาด/กราฟิก",
}
MODEL_REF = "a00d55fee90e0f9435d5eaa86e76700df8199af8"
MODEL_BASE_URL = (
    "https://raw.githubusercontent.com/upscayl/upscayl/"
    f"{MODEL_REF}/resources/models"
)
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp"}


def ensure_model(model_id):
    for suffix, min_bytes in ((".param", 1_000), (".bin", 100_000)):
        target = MODEL_DIR / f"{model_id}{suffix}"
        if target.is_file() and target.stat().st_size >= min_bytes:
            continue
        if target.exists():
            target.unlink()
        download_file(f"{MODEL_BASE_URL}/{model_id}{suffix}", target)
        if not target.is_file() or target.stat().st_size < min_bytes:
            raise RuntimeError(f"ดาวน์โหลดโมเดล {target.name} ไม่สมบูรณ์")


def get_input_path(file_item):
    if isinstance(file_item, (str, Path)):
        return Path(str(file_item))
    if isinstance(file_item, dict):
        value = file_item.get("path") or file_item.get("name")
    else:
        value = getattr(file_item, "path", None) or getattr(file_item, "name", None)
    if not value:
        raise ValueError(f"อ่าน path ของไฟล์ไม่ได้: {file_item!r}")
    return Path(value)


def run_upscayl(files_in, model_id, scale, output_format, tile_size, use_tta, include_input, progress=gr.Progress()):
    logs = []
    if not globals().get("BACKEND_READY", False):
        reason = globals().get("BACKEND_ERROR") or "ยังไม่ได้เตรียม backend"
        return [], None, f"### Backend ยังไม่พร้อม\n{reason}", "Backend setup ยังไม่สำเร็จ"

    job_id = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
    staged_uploads = RUNTIME_DIR / "gradio-inputs" / job_id
    job_output = OUTPUT_DIR / job_id
    staged_uploads.mkdir(parents=True, exist_ok=True)
    job_output.mkdir(parents=True, exist_ok=True)

    jobs = []
    used_names = set()
    for index, item in enumerate(files_in or [], start=1):
        try:
            source = get_input_path(item)
            if not source.is_file() or source.suffix.lower() not in IMAGE_EXTENSIONS:
                logs.append(f"ข้ามไฟล์ที่ไม่รองรับ: {source}")
                continue
            filename = source.name.replace("\\", "/").split("/")[-1]
            if filename.lower() in used_names:
                filename = f"{index}_{filename}"
            used_names.add(filename.lower())
            staged = staged_uploads / filename
            shutil.copy2(source, staged)
            jobs.append((staged, Path(filename), "upload"))
        except Exception as exc:
            logs.append(f"อ่านไฟล์ที่อัปโหลดไม่ได้: {exc}")

    if include_input:
        for source in sorted(INPUT_DIR.rglob("*")):
            if source.is_file() and source.suffix.lower() in IMAGE_EXTENSIONS and not source.name.startswith("."):
                jobs.append((source, source.relative_to(INPUT_DIR), "input"))

    if not jobs:
        return [], None, "### กรุณาเลือกรูปก่อนเริ่ม", "\n".join(logs) or "ยังไม่มีไฟล์ภาพ"

    progress(0, desc="เตรียมโมเดล...")
    try:
        ensure_model(model_id)
    except Exception as exc:
        return [], None, f"### ดาวน์โหลดโมเดลไม่สำเร็จ\n{exc}", "\n".join(logs)

    results = []
    successes = []
    failures = []
    for index, (source, relative, source_kind) in enumerate(jobs, start=1):
        output_folder = job_output / source_kind / relative.parent
        output_folder.mkdir(parents=True, exist_ok=True)
        destination = output_folder / (
            f"{relative.stem}_upscayl_{scale}x_{model_id}.{output_format}"
        )
        command = [
            str(BIN_PATH),
            "-i", str(source),
            "-o", str(destination),
            "-s", str(scale),
            "-m", str(MODEL_DIR),
            "-n", model_id,
            "-g", "0",
            "-t", str(tile_size),
            "-f", output_format,
            "-v",
        ]
        if use_tta:
            command.append("-x")
        progress((index - 1) / len(jobs), desc=f"Upscale {index}/{len(jobs)} · {relative.name}")
        try:
            result = subprocess.run(
                command,
                text=True,
                capture_output=True,
                env=os.environ.copy(),
            )
            log = (result.stdout or "") + (result.stderr or "")
        except Exception as exc:
            log = str(exc)
            result = None

        if result is None or result.returncode != 0 or not destination.is_file() or destination.stat().st_size == 0:
            failures.append(relative.name)
            logs.append(f"❌ {relative.name}\n{log[-1800:]}")
            continue

        successes.append((source, destination, source_kind))
        source_label = "Input" if source_kind == "input" else "Original"
        results.extend([
            (str(source), f"{source_label} · {relative.name}"),
            (str(destination), f"Upscaled · {destination.name}"),
        ])
        logs.append(f"✅ {relative.name} → {destination.name}")
        progress(index / len(jobs), desc=f"เสร็จ {index}/{len(jobs)}")

    if not successes:
        return [], None, f"### Upscale ไม่สำเร็จ ({len(failures)} ไฟล์)", "\n\n".join(logs[-20:])

    archive_path = RUNTIME_DIR / f"upscayl-results-{job_id}.zip"
    with zipfile.ZipFile(archive_path, "w", compression=zipfile.ZIP_STORED) as archive:
        for _, result_path, _ in successes:
            archive.write(result_path, arcname=str(result_path.relative_to(job_output)))

    message = f"### เสร็จแล้ว {len(successes)}/{len(jobs)} ภาพ"
    if failures:
        message += f"\n\nมี {len(failures)} ภาพที่ล้มเหลว ดูรายละเอียดใน log"
    logs.append(f"\nZIP: {archive_path}")
    return results, str(archive_path), message, "\n\n".join(logs[-20:])


CSS = """
.gradio-container { max-width: 1180px !important; margin: 0 auto !important; }
#upscayl-hero { background: linear-gradient(120deg,#111827,#312e81 65%,#4f46e5); color: #fff; padding: 26px 30px; border-radius: 20px; margin: 10px 0 20px; }
#upscayl-hero h1 { color: #fff; font-size: 30px; margin: 8px 0; }
#upscayl-hero p { color: #dbeafe; margin: 0; }
#upscayl-panel { border: 1px solid #e2e8f0; border-radius: 18px; padding: 18px; }
#upscayl-run button { min-height: 48px; font-weight: 700; }
#upscayl-output { border: 1px solid #e2e8f0; border-radius: 18px; padding: 12px; }
footer { display: none !important; }
"""

with gr.Blocks(title="Upscayl Studio", css=CSS) as demo:
    gr.HTML(
        '<div id="upscayl-hero"><div style="font-size:12px;letter-spacing:2px;opacity:.82">NCNN · VULKAN · COLAB GPU</div>'
        '<h1>Upscayl Studio</h1><p>อัปสเกลภาพด้วย Upscayl รุ่นทางการ · รองรับประมวลผลหลายภาพ</p></div>'
    )
    with gr.Row():
        with gr.Column(scale=4, min_width=300, elem_id="upscayl-panel"):
            gr.Markdown("### 1 · เลือกภาพ")
            input_files = gr.File(
                label="ลากภาพมาวาง หรือกด Browse files",
                file_count="multiple",
                file_types=["image"],
                type="filepath",
            )
            include_input = gr.Checkbox(
                label="รวมรูปจากโฟลเดอร์ input (เช่น Google Drive)",
                value=False,
            )
            gr.Markdown("### 2 · ตั้งค่า")
            model = gr.Dropdown(
                choices=[(label, model_id) for model_id, label in MODEL_LABELS.items()],
                value="upscayl-standard-4x",
                label="AI Model",
            )
            with gr.Row():
                scale = gr.Dropdown(
                    choices=[("2×", 2), ("3×", 3), ("4×", 4)],
                    value=4,
                    label="Output scale",
                )
                output_format = gr.Dropdown(
                    choices=[("PNG", "png"), ("JPG", "jpg"), ("WebP", "webp")],
                    value="png",
                    label="Format",
                )
            tile_size = gr.Dropdown(
                choices=[("Auto", 0), ("32 · VRAM ต่ำ", 32), ("64", 64), ("128", 128)],
                value=0,
                label="Tile size",
            )
            use_tta = gr.Checkbox(label="เปิด TTA (ช้าลง แต่อาจเพิ่มรายละเอียด)", value=False)
            run_button = gr.Button("เริ่ม Upscale", variant="primary", elem_id="upscayl-run")
            zip_file = gr.File(label="ดาวน์โหลดผลลัพธ์ ZIP", interactive=False)
        with gr.Column(scale=6, min_width=360):
            initial_status = (
                "### Backend พร้อมใช้งาน\nเลือกภาพและตั้งค่าจากแผงด้านซ้าย"
                if globals().get("BACKEND_READY", False)
                else f"### UI พร้อมแล้ว แต่ backend ยังไม่พร้อม\n{globals().get('BACKEND_ERROR') or 'ดู log ของเซลล์ backend ด้านบน'}"
            )
            status = gr.Markdown(initial_status)
            gallery = gr.Gallery(
                label="เปรียบเทียบก่อน · หลัง",
                columns=2,
                height=520,
                object_fit="contain",
                elem_id="upscayl-output",
            )
            logs = gr.Textbox(label="บันทึกการทำงาน", lines=8, max_lines=14, interactive=False)

    run_button.click(
        fn=run_upscayl,
        inputs=[input_files, model, scale, output_format, tile_size, use_tta, include_input],
        outputs=[gallery, zip_file, status, logs],
        show_progress="full",
    )

# A temporary, password-protected browser URL gives a proper web UI in Colab.
LOGIN_USER = "upscayl"
LOGIN_PASSWORD = secrets.token_urlsafe(12)
print("กำลังเปิด Upscayl Studio ใน browser...")
print("ลิงก์เป็นชั่วคราวและป้องกันด้วยรหัสผ่าน; หยุด Colab runtime เพื่อปิดลิงก์")
demo.queue().launch(
    server_name="0.0.0.0",
    share=True,
    auth=(LOGIN_USER, LOGIN_PASSWORD),
    auth_message="Sign in with the credentials printed in Colab.",
    show_error=True,
    inline=True,
    prevent_thread_lock=True,
)
print(f"Username: {LOGIN_USER}\nPassword: {LOGIN_PASSWORD}")


### แหล่งที่มาและลิขสิทธิ์

Notebook ดาวน์โหลด Upscayl NCNN backend และคู่ไฟล์โมเดลจาก upstream ขณะใช้งาน โดยตรึงเวอร์ชัน CLI และ model commit ไว้ ดูข้อมูล upstream และเงื่อนไขการใช้งานที่ [README](README.md)
